# Task 3 — Unsupervised Learning and Clustering
## UCI Adult Income Dataset

**Objective:** Apply an unsupervised learning technique to segment the UCI Adult dataset into meaningful groups and analyze the characteristics of the resulting clusters.

**Algorithm:** K-Means Clustering  
**Libraries:** Pandas, NumPy, Matplotlib, Seaborn, Scikit-learn  
**Dataset:** UCI Adult (Census Income) Dataset

This task continues the same public dataset used in Tasks 1 and 2. The `income` target is retained only for **post-clustering descriptive analysis** and is **never used as an input to K-Means**.

## 1. Import Libraries and Load the Dataset

The notebook uses the UCI Adult dataset (UCI dataset ID 2). The `ucimlrepo` package is used to retrieve the public dataset. If the package is not installed in the current Jupyter environment, the next cell installs it automatically.

> **Important:** `fetch_ucirepo` comes from `ucimlrepo`, not from `sklearn.datasets`.

In [ ]:
# Install the dataset-access package only if it is missing
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("ucimlrepo") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "ucimlrepo"
    ])

# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# UCI dataset access
from ucimlrepo import fetch_ucirepo

# Scikit-learn tools
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

# Reproducibility and plotting
RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

In [ ]:
# Load the UCI Adult dataset
adult = fetch_ucirepo(id=2)

X = adult.data.features.copy()
y = adult.data.targets.copy()

# Clean column names and string values
X.columns = [str(c).strip() for c in X.columns]

for col in X.select_dtypes(include=["object", "string"]).columns:
    X[col] = X[col].astype("string").str.strip()

target_name = y.columns[0]
y[target_name] = (
    y[target_name]
    .astype("string")
    .str.strip()
    .str.replace(".", "", regex=False)
)

# Combine features and target for later descriptive profiling
df = X.copy()
df["income"] = y[target_name].astype(str).values

print("Dataset shape:", df.shape)
print("\nFeature columns:")
print(X.columns.tolist())
print("\nTarget distribution:")
print(df["income"].value_counts())

## 2. Inspect the Data

Before clustering, the feature types and missing-value markers are checked. In the Adult dataset, missing categorical observations can be represented by `?`.

In [ ]:
print("Data types:")
display(X.dtypes)

print("\nMissing-value markers before treatment:")
for col in X.columns:
    count_q = X[col].astype(str).str.strip().eq("?").sum()
    if count_q > 0:
        print(f"{col}: {count_q}")

## 3. Data Preparation for Clustering

K-Means requires numerical input. The following steps are applied:

1. Replace `?` markers with missing values.
2. Impute numerical missing values with the median.
3. Impute categorical missing values with the mode.
4. Apply `log1p` to the highly right-skewed, non-negative `capital-gain` and `capital-loss` variables.
5. Standardize numerical features.
6. One-hot encode categorical features.

The `income` target is excluded from the clustering matrix.

In [ ]:
# Replace Adult dataset missing-value markers with NaN
X_clean = X.copy()
X_clean = X_clean.replace({"?": np.nan, " ?": np.nan})

# Identify feature types
numeric_cols = X_clean.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X_clean.select_dtypes(exclude=np.number).columns.tolist()

print("Numerical features:", numeric_cols)
print("Categorical features:", categorical_cols)

print("\nMissing values after replacing '?':")
missing_counts = X_clean.isna().sum()
display(missing_counts[missing_counts > 0].sort_values(ascending=False))

print("Total missing cells:", int(X_clean.isna().sum().sum()))

In [ ]:
# Impute missing values
for col in numeric_cols:
    X_clean[col] = X_clean[col].fillna(X_clean[col].median())

for col in categorical_cols:
    mode_values = X_clean[col].mode(dropna=True)
    fill_value = mode_values.iloc[0] if not mode_values.empty else "Unknown"
    X_clean[col] = X_clean[col].fillna(fill_value)

print("Remaining missing cells:", int(X_clean.isna().sum().sum()))

## 4. Transform and Encode the Features

K-Means is distance-based, so numerical variables with large scales must not dominate the distance calculation. Standardization puts numerical variables on comparable scales.

`capital-gain` and `capital-loss` contain many zeros and a small number of very large values. `log1p(x)` reduces their skewness while preserving zero as zero.

In [ ]:
X_model = X_clean.copy()

# Reduce skewness of the two highly skewed non-negative variables
for col in ["capital-gain", "capital-loss"]:
    X_model[col] = np.log1p(X_model[col].clip(lower=0))

# Create a version-compatible OneHotEncoder
try:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    # Compatibility with older scikit-learn versions
    encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", encoder, categorical_cols)
    ]
)

X_encoded = preprocessor.fit_transform(X_model)

print("Processed feature matrix shape:", X_encoded.shape)
print("Processed feature matrix contains NaN:", bool(np.isnan(X_encoded).any()))
print("Processed feature matrix contains infinite values:", bool(np.isinf(X_encoded).any()))

## 5. Choose the Number of Clusters

The notebook evaluates **K = 2, 3, 4, 5, and 6**.

- **Inertia:** lower values indicate more compact clusters; it is examined using the elbow method.
- **Silhouette score:** higher values indicate better separation and cohesion.

A fixed random sample of up to 5,000 observations is used for model selection to reduce computation time. The final model is subsequently fitted to the complete dataset.

In [ ]:
k_values = list(range(2, 7))
inertias = []
silhouette_scores = []

rng = np.random.RandomState(RANDOM_STATE)
sample_size = min(5000, X_encoded.shape[0])
sample_idx = rng.choice(X_encoded.shape[0], size=sample_size, replace=False)
X_sample = X_encoded[sample_idx]

for k in k_values:
    model = KMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        n_init=10
    )
    sample_labels = model.fit_predict(X_sample)

    inertias.append(model.inertia_)
    silhouette_scores.append(
        silhouette_score(X_sample, sample_labels)
    )

results_k = pd.DataFrame({
    "K": k_values,
    "Inertia": inertias,
    "Silhouette Score": silhouette_scores
})

display(results_k.round(4))

In [ ]:
# Elbow plot
plt.figure(figsize=(8, 5))
plt.plot(results_k["K"], results_k["Inertia"], marker="o")
plt.xlabel("Number of clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method for K-Means")
plt.xticks(k_values)
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

# Silhouette plot
plt.figure(figsize=(8, 5))
plt.plot(results_k["K"], results_k["Silhouette Score"], marker="o")
plt.xlabel("Number of clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Analysis")
plt.xticks(k_values)
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

best_k = int(results_k.loc[results_k["Silhouette Score"].idxmax(), "K"])
print("Selected K based on the highest silhouette score:", best_k)

## 6. Fit the Final K-Means Model

The K value with the highest silhouette score is selected. The final K-Means model is fitted to **all processed observations**.

In [ ]:
kmeans = KMeans(
    n_clusters=best_k,
    random_state=RANDOM_STATE,
    n_init=10
)

cluster_labels = kmeans.fit_predict(X_encoded)

df_clustered = df.copy()
df_clustered["Cluster"] = cluster_labels

cluster_counts = (
    df_clustered["Cluster"]
    .value_counts()
    .sort_index()
)

print("Cluster sizes:")
display(cluster_counts.to_frame("Observations"))

print("Final K-Means inertia:", round(kmeans.inertia_, 2))

## 7. Visualize the Clusters Using PCA

One-hot encoding creates a high-dimensional feature space. PCA reduces this space to two principal components **only for visualization**.

The K-Means model itself is fitted using the complete processed feature matrix, not the two PCA components.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_encoded)

plot_df = pd.DataFrame({
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1],
    "Cluster": cluster_labels
})

plot_sample = plot_df.sample(
    n=min(6000, len(plot_df)),
    random_state=RANDOM_STATE
)

plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=plot_sample,
    x="PC1",
    y="PC2",
    hue="Cluster",
    palette="tab10",
    alpha=0.55,
    s=35
)
plt.title("K-Means Clusters in PCA Space")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(title="Cluster")
plt.tight_layout()
plt.show()

print("Explained variance ratio:", np.round(pca.explained_variance_ratio_, 4))
print(
    "Total variance explained by two PCs:",
    round(float(pca.explained_variance_ratio_.sum()), 4)
)

## 8. Numerical Cluster Profiles

Means and medians are calculated for the original numerical variables. Comparing both helps because variables such as capital gain and capital loss are strongly skewed.

In [ ]:
profile_numeric = (
    df_clustered
    .groupby("Cluster")[numeric_cols]
    .agg(["mean", "median"])
    .round(2)
)

display(profile_numeric)

## 9. Income Composition by Cluster

The `income` variable was **not used to form the clusters**. It is examined here only as an external descriptive variable to understand how the discovered segments differ in their observed income composition.

The percentages below are calculated **within each cluster**, so each row sums to approximately 100%.

In [ ]:
income_profile = pd.crosstab(
    df_clustered["Cluster"],
    df_clustered["income"],
    normalize="index"
).mul(100).round(2)

print("Income composition by cluster (%):")
display(income_profile)

ax = income_profile.plot(
    kind="bar",
    stacked=True,
    figsize=(9, 5)
)
ax.set_title("Income Composition Across Clusters")
ax.set_xlabel("Cluster")
ax.set_ylabel("Percentage of observations")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title="Income")
plt.tight_layout()
plt.show()

## 10. Categorical Cluster Profiles

For each categorical feature, the most common category within every cluster is reported together with its percentage. This provides a more informative profile than listing only the category name.

In [ ]:
categorical_summary = []

for col in categorical_cols:
    grouped = (
        df_clustered.groupby("Cluster")[col]
        .value_counts(normalize=True)
        .mul(100)
        .rename("Percentage")
        .reset_index()
    )

    top_rows = (
        grouped.sort_values(
            ["Cluster", "Percentage"],
            ascending=[True, False]
        )
        .groupby("Cluster", as_index=False)
        .head(1)
    )

    top_rows["Feature"] = col
    top_rows = top_rows.rename(columns={col: "Most Common Category"})
    categorical_summary.append(
        top_rows[["Feature", "Cluster", "Most Common Category", "Percentage"]]
    )

categorical_profile = pd.concat(
    categorical_summary,
    ignore_index=True
)

categorical_profile["Percentage"] = categorical_profile["Percentage"].round(2)

display(categorical_profile)

## 11. Cluster Size Visualization

Cluster sizes are checked to determine whether one cluster dominates the segmentation or whether the observations are distributed across several groups.

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(
    x=cluster_counts.index.astype(str),
    y=cluster_counts.values
)
plt.title("Number of Observations in Each Cluster")
plt.xlabel("Cluster")
plt.ylabel("Number of observations")
plt.tight_layout()
plt.show()

display(cluster_counts.to_frame("Observations"))

## 12. Compact Cluster Comparison

This table provides a concise summary for the final report. The cluster labels themselves have no inherent meaning: **Cluster 0 is not automatically a low-income group, and another cluster is not automatically a high-income group.**

Interpretation must be based on the numerical and categorical profiles generated above.

In [ ]:
summary_cols = [
    c for c in [
        "age",
        "education-num",
        "hours-per-week",
        "capital-gain",
        "capital-loss"
    ]
    if c in df_clustered.columns
]

cluster_summary = (
    df_clustered.groupby("Cluster")[summary_cols]
    .mean()
    .round(2)
)

cluster_summary["Size"] = (
    df_clustered.groupby("Cluster")
    .size()
)

cluster_summary["Size (%)"] = (
    cluster_summary["Size"]
    .div(len(df_clustered))
    .mul(100)
    .round(2)
)

display(cluster_summary)

## 13. Interpretation Guide

Use the actual outputs above to describe each cluster. For every cluster, compare:

- average and median age,
- education level (`education-num`),
- weekly working hours,
- capital gain and capital loss,
- dominant workclass,
- dominant occupation,
- dominant marital status and relationship,
- and descriptive income composition.

The interpretation should describe **associations in this historical dataset**, not causal relationships. Cluster membership also should not be treated as a prediction or as proof that a particular characteristic causes income differences.

## 14. Conclusion

K-Means clustering was applied after missing-value treatment, skewness reduction, standardization, and one-hot encoding. The number of clusters was assessed using the elbow method and silhouette score. PCA was then used to visualize the high-dimensional clustering result.

The final cluster profiles provide an unsupervised segmentation of the Adult dataset. The observed differences can support exploratory socioeconomic research, workforce segmentation, and further analysis. Because clustering is unsupervised, the discovered groups should be interpreted from their actual profiles rather than assigned meanings in advance.